# Experiment 1: Baseline

Compare retrieval and generated answers on the same 31 seeded eligible questions and fixed corpus, with separately saved diagnostic examples.

**Saved execution:** the outputs below preserve the recorded Qwen run. Opening this notebook does not require rerunning its cells. For deliberate replay, set `NOTEBOOK_MODE=replay` before starting the kernel. Execute mode can prepare missing work when paid calls are explicitly enabled through `NOTEBOOK_ALLOW_MODEL_CALLS=1` and `OPENROUTER_API_KEY`; judging has additional action controls. This editorial cleanup did not execute any cells or change model settings.

**Study scope:** the frozen larger corpus contains **152,576 passages** associated with **15,585 fully covered question IDs** from MS MARCO v2.1 validation; this is a subset, not the full validation split. Experiment 1 uses a separate **10,210-passage corpus**, 31 seeded baseline questions, the original saved effervescent diagnostic, and a separately saved IMEI diagnostic. Experiment 2 samples **500 opportunities + 200 controls**. Experiment 3 contains **79 sampled pairs, including IMEI**. Experiment 4 times **8 question pairs / 16 answers**. IMEI is excluded from the 31-question baseline averages but is already included in the frozen Experiment 2 and Experiment 3 samples; displaying it again adds no observation.

**Workflow:** **this notebook Experiment 1 (Baseline)** --> Diagnosis --> Experiment 2 (Targeted Reranking) --> Experiment 3 (Answer Quality) --> Experiment 4 (Latency Tradeoff)


## 1. Experiment setup

The table shows the main models and passage budgets. Expand the configuration below it for generation settings, endpoints, retry policies and the system prompt.

`configs/experiment1.json` selects the saved run and scoring settings. The pipeline configuration is loaded from that run’s `config.json`.


In [1]:
from pathlib import Path
import os, json, time, hashlib, math, re, string, platform, importlib.metadata, inspect, sys
from datetime import datetime, timezone
from collections import Counter
import numpy as np
import pandas as pd
from IPython.display import display, Markdown, Code

PROJECT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                if (p / 'configs/experiment1.json').is_file()), None)
if PROJECT is None:
    raise FileNotFoundError('Open this notebook within the project containing configs/experiment1.json.')
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
from shared_scoring import load_configuration, load_saved_scores, load_evaluation_logs
from shared_scoring.io import read_json
from shared_scoring.views import show_question, show_saved_request
from shared_scoring.execution import ensure_baseline_study
from shared_scoring.notebook_runtime import NotebookSession, StudyNotReady
session = NotebookSession(PROJECT)
MODE = session.mode
standard_config = session.saved_configuration('baseline')
if MODE == 'execute':
    if standard_config is None:
        session.preflight('baseline')  # Tests before the first missing paid request.
    # Actual query embedding, retrieval, reranking and generation run on first execution.
    # A completed matching run is reused on later executions.
    standard_config = ensure_baseline_study(load_configuration(PROJECT / 'configs/experiment1.json'), PROJECT)
elif standard_config is None:
    raise StudyNotReady('No saved Qwen baseline is available for replay.')
ROOT = standard_config['source_directory']
INPUT = ROOT / 'input'
REFERENCE = standard_config.get('historical_reference_directory', ROOT / 'reference')
RUN = ROOT / 'runs' / standard_config['run_id']
settings = read_json(RUN / 'config.json')
cfg, SYSTEM = settings['models_and_dataset'], settings['system_prompt']
GENERATION_EXTRAS = settings['generation_extras']
VERSION = 'v1-baseline'
BATCH_SIZE = cfg['embedding_batch_size']
ARMS = tuple(standard_config['arms'])
DISPLAY_NAMES = standard_config['arms']
DISPLAY_QUERY_ID = standard_config['diagnostic_ids'][0]
identity = read_json(ROOT / 'batch_identity_v1.json')
assert identity['run_id'] == standard_config['run_id']
execution_mode, BATCH_COMPLETE = 'recorded_batch', True

def digest(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False).encode()).hexdigest()

def file_hash(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

pd.set_option('display.max_colwidth', 160)
pd.set_option('display.max_rows', 80)
def experiment_snapshot(config):
    return pd.DataFrame([
        ("Dataset", f"{config['dataset']} · {config['dataset_config']} · {config['split']}"),
        ("Embedding model", config['embedding_model']),
        ("Reranker model", config['reranker_model']),
        ("Generator model", config['generator_model']),
        ("Candidate / context budget", f"{config['top_candidates']} / {config['top_context']} passages"),
        ("Passage text limit", f"{config['passage_char_budget']:,} characters per passage"),
        ("Generation output limit", f"{config['max_output_tokens']} tokens"),
    ], columns=["Setting", "Value"])

display(experiment_snapshot(cfg))
from html import escape
from IPython.display import HTML
configuration_details = {
    'pipeline': cfg,
    'generation_extras': GENERATION_EXTRAS,
    'system_prompt': SYSTEM,
    'run_id': identity['run_id'],
}
display(HTML('<details><summary>Full configuration and system prompt</summary><pre>'
             + escape(json.dumps(configuration_details, indent=2, ensure_ascii=False))
             + '</pre></details>'))
print('Fresh run prepared; inspect its request records. RAGAS remains an explicit action.'
      if standard_config.get('fresh_run') else 'Saved-results walkthrough; no credentials loaded and no model calls.')


Saved baseline pipeline artifacts are missing; preparing a separate fresh baseline.


Reused 10,210 cached passage embeddings for the eligible baseline; no passage embedding calls.


.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,Setting,Value
0,Dataset,microsoft/ms_marco · v2.1 · validation
1,Embedding model,qwen/qwen3-embedding-4b
2,Reranker model,qwen/qwen3-reranker-8b
3,Generator model,qwen/qwen3-30b-a3b-instruct-2507
4,Candidate / context budget,30 / 5 passages
5,Passage text limit,"1,800 characters per passage"
6,Generation output limit,400 tokens


Fresh run prepared; inspect its request records. RAGAS remains an explicit action.


## 2. Verify the 31 seeded eligible questions

The selected question list, saved candidates, saved results and frozen IDs must agree. Inspect the questions below; the code verifies their order and membership.

The preceding source run’s completion counts are shown separately. They are provenance, not the completion count of the baseline being evaluated here.


In [2]:
baseline_ids = json.loads((INPUT / 'baseline_ids.json').read_text())
reviewed = [r for r in json.loads((INPUT / 'golden_reviewed.json').read_text())
            if str(r['decision']).strip().lower() == 'keep']
prior_questions = (json.loads((REFERENCE / 'prior_candidates.json').read_text())['questions']
                   if (REFERENCE / 'prior_candidates.json').is_file() else [])
prior_results = (json.loads((REFERENCE / 'prior_results.json').read_text())
                 if (REFERENCE / 'prior_results.json').is_file() else [])
if not prior_questions or not prior_results:
    print('Optional historical comparison files are absent; the current study remains runnable.')
assert len(baseline_ids) == len(set(baseline_ids)) == 31
assert baseline_ids == [str(r['query id']) for r in reviewed]
assert not prior_questions or baseline_ids == [q['query_id'] for q in prior_questions]
assert not prior_results or baseline_ids == [r['query_id'] for r in prior_results]
assert DISPLAY_QUERY_ID not in baseline_ids

prior_pair_ids = [r['query_id'] for r in prior_results
                  if all(r['arms'][a]['status'] == 'ok' for a in ARMS)]
prior_failures = [{'query_id': r['query_id'], 'arm': a, 'error': v.get('error')}
                  for r in prior_results for a, v in r['arms'].items() if v['status'] != 'ok']
display(pd.DataFrame([{'prior_attempted_questions': len(prior_results),
                      'prior_successful_pairs': len(prior_pair_ids),
                      'prior_failed_arms': len(prior_failures)}]))
display(pd.DataFrame(prior_failures))
display(pd.DataFrame({'baseline_position': range(1, 32), 'query_id': baseline_ids,
                      'question': [r['query'] for r in reviewed]}))


Optional historical comparison files are absent; the current study remains runnable.


,prior_attempted_questions,prior_successful_pairs,prior_failed_arms
0,0,0,0


""


,baseline_position,query_id,question
0,1,1088186,"weather in texarkana, arkansas"
1,2,1038185,what is the ideal credit card balance for credit score
2,3,1086238,what causes antibiotic resistance in bacteria
3,4,1100277,gestation medical definition
4,5,1029544,what is tracers
5,6,1082398,what does warbled mean
6,7,1055197,what is flex fuel on a jeep?
7,8,1089273,types of road hugger tires
8,9,1085733,what could cause soreness of heel
9,10,1085627,what county is charleston ar in?


## 3. Load the questions and fixed corpus

The baseline corpus contains **10,210 deduplicated passages** associated with the **31 seeded baseline questions** and **1,000 additional distractor questions**. Distractor questions were sampled with seed 42 from the same 15,585 eligible IDs, excluding the baseline questions and the original effervescent diagnostic. All associated selected and unselected passages were included; individual distractor passages were not sampled independently and need not be irrelevant to every baseline question.

The dataset revision is pinned. Passages are trimmed, deduplicated by case-insensitive text, assigned stable IDs, and limited to 1,800 characters. The reconstructed corpus must match the saved corpus.

The IMEI diagnostic **1053665 (what is imei and where do i find it)** is evaluated separately against this unchanged corpus, which contains **0 of its 2 annotated supporting passages**. Its answers and scores are saved separately from the original batch. The original saved batch ledger still contains **1100870 (effervescent meaning)** as its diagnostic; those historical records are not relabeled as IMEI.


In [3]:
source_metadata = json.loads((INPUT / 'source_metadata.json').read_text())
assert source_metadata['Source revision'] == cfg['revision']
rows = json.loads((INPUT / 'dataset_rows.json').read_text())
diagnostic_row = json.loads((INPUT / f'diagnostic_{DISPLAY_QUERY_ID}.json').read_text())
by_id = {str(r['query_id']): r for r in rows}
assert len(rows) == len(by_id) == 1031
assert [str(r['query_id']) for r in rows[:31]] == baseline_ids
assert str(diagnostic_row['query_id']) == DISPLAY_QUERY_ID
assert diagnostic_row['query'].strip()

def passage_id(text):
    return 'p_' + digest(text.strip().casefold())[:20]

raw_docs = {}
for row in rows:
    for text, url in zip(row['passages']['passage_text'], row['passages']['url']):
        text = text.strip()
        if not text:
            continue
        pid = passage_id(text)
        raw_docs.setdefault(pid, {'id': pid, 'text': text, 'urls': []})
        if url not in raw_docs[pid]['urls']:
            raw_docs[pid]['urls'].append(url)
docs = [dict(d, original_char_count=len(d['text']),
             truncated=len(d['text']) > cfg['passage_char_budget'],
             text=d['text'][:cfg['passage_char_budget']])
        for d in sorted(raw_docs.values(), key=lambda d: d['id'])]
assert docs == json.loads((INPUT / 'corpus.json').read_text())
assert len(docs) > 0
lookup = {d['id']: d for d in docs}

golden = []
for record in reviewed:
    qid = str(record['query id'])
    row = by_id[qid]
    assert row['query'].strip() == record['query'].strip()
    answers = json.loads(record['answer'])
    assert answers == row['answers'] and answers and all(a.strip() for a in answers)
    blocks = re.split(r'(?:^|\n\n)Passage \d+\n', record['related passages'])[1:]
    reviewed_passages = []
    for block in blocks:
        text, url = block.rsplit('\nSource: ', 1)
        reviewed_passages.append({'text': text.strip(), 'url': url.strip()})
    gold_ids = sorted({passage_id(p['text']) for p in reviewed_passages})
    selected = {passage_id(t) for t, flag in zip(row['passages']['passage_text'],
                row['passages']['is_selected']) if flag}
    assert set(gold_ids) <= selected and set(gold_ids) <= lookup.keys()
    golden.append({'query_id': qid, 'query': record['query'], 'answers': answers,
                   'review_notes': record['why'], 'reviewed_passages': reviewed_passages,
                   'gold_ids': gold_ids, 'scope': 'baseline_31'})

diagnostic_gold = sorted({passage_id(t) for t, selected in zip(
    diagnostic_row['passages']['passage_text'], diagnostic_row['passages']['is_selected']) if selected})
diagnostic = {'query_id': DISPLAY_QUERY_ID, 'query': diagnostic_row['query'],
    'answers': diagnostic_row['answers'], 'gold_ids': diagnostic_gold, 'scope': 'diagnostic',
    'review_notes': 'Dataset selected support, outside the original reviewed 31. Excluded from baseline aggregates.',
    'reviewed_passages': [{'text': t.strip(), 'url': u.strip()} for t, u, selected in zip(
        diagnostic_row['passages']['passage_text'], diagnostic_row['passages']['url'],
        diagnostic_row['passages']['is_selected']) if selected]}
batch_questions = golden + [diagnostic]
assert len(batch_questions) == 32 and [q['query_id'] for q in golden] == baseline_ids
display(pd.DataFrame([{'dataset': cfg['dataset'], 'revision': cfg['revision'],
    'baseline_questions': len(golden), 'diagnostic_questions': 1, 'corpus_passages': len(docs),
    'clipped_passages': sum(d['truncated'] for d in docs),
    'diagnostic_gold_in_corpus': len(set(diagnostic_gold) & lookup.keys()),
    'diagnostic_gold_total': len(diagnostic_gold)}]))
display(pd.DataFrame([{'query_id': q['query_id'], 'scope': q['scope'],
    'question': q['query'], 'reference_answers': q['answers'], 'gold_ids': q['gold_ids']}
    for q in batch_questions]))


,dataset,revision,baseline_questions,diagnostic_questions,corpus_passages,clipped_passages,diagnostic_gold_in_corpus,diagnostic_gold_total
0,microsoft/ms_marco,a47ee7aae8d7d466ba15f9f0bfac3b3681087b3a,31,1,10210,0,0,1


,query_id,scope,question,reference_answers,gold_ids
0,1088186,baseline_31,"weather in texarkana, arkansas",[The Texarkana forecast for Aug 30 is 76 degrees and Patchy rain possible. There is 83 percentage chance of rain and 13 mph winds from the Northeast. The Te...,[p_10af63d165c7c485aedb]
1,1038185,baseline_31,what is the ideal credit card balance for credit score,"[If you have a credit card with a $1,000 credit limit and a $100 balance then you are 10% “utilized” on that card.]",[p_40ecf58fe3664fcffaa5]
2,1086238,baseline_31,what causes antibiotic resistance in bacteria,"[If you take an antibiotic when you actually have a viral infection, the antibiotic attacks bacteria in your body bacteria that are either beneficial or at ...",[p_32a99c36bfc8d9cebdca]
3,1100277,baseline_31,gestation medical definition,[Gestation is defined as the period of fetal development from conception until birth.],[p_ded14ec74054e77e2a93]
4,1029544,baseline_31,what is tracers,[Tracers is a 2015 American action film.],[p_8c691bd8eb091d4fefd6]
5,1082398,baseline_31,what does warbled mean,"[Warble means to sing or whistle with trills, quavers, or melodic embellishments.]",[p_e11e11ab1f92dca78f46]
6,1055197,baseline_31,what is flex fuel on a jeep?,[Jeeps display a silver Flex Fuel E85 tag. E85 ethanol flex-fuel vehicles from Mercedes-Benz and Nissan do not carry any special identification.],[p_9e32bd1a8c1c2459bc87]
7,1089273,baseline_31,types of road hugger tires,[Road Hugger Radial GT is a type of Road Hugger tire.],[p_b7ef8ac8a31403454616]
8,1085733,baseline_31,what could cause soreness of heel,"[The causes of heel pain are plantar fasciitis, achilles tendinitis, achilles tendon rupture, bone tumor, bursitis Haglund's deformity, heel spur, osteomyel...",[p_daed0067e95cbd258615]
9,1085627,baseline_31,what county is charleston ar in?,"[Charleston is in Franklin County, Arkansas.]",[p_7d632d8c7a3160bec3ce]


## 4. Document embedding and cache

The baseline reuses existing cached Qwen document embeddings for its **10,210 passages**. The saved matrix, origin record, and configuration document that reuse; no new document embedding call is needed to inspect this run.

The functions below expose the embedding interface and provenance. An empty document-attempt table reflects the saved records available to this baseline, not proof that the documents lack vectors.


In [4]:
from shared_scoring.fresh_retrieval import RecordedOpenRouterClient, execution_settings

def embed_document_batch(client, documents):
    # OpenRouter embedding request; successful matching responses are cached.
    return client.embed([d['text'] for d in documents], 'passage', cfg['embedding_dimensions'])

document_origin = read_json(RUN / 'document_embedding_origin.json')
document_cache_id = document_origin['cache_id']
# The first execution created this matrix; later execution uses its saved vectors.
document_matrix = np.load(ROOT / 'document_embeddings.npy', mmap_mode='r')
assert document_matrix.shape == (len(docs), cfg['embedding_dimensions'])
display(document_origin)
print('Saved document embedding matrix:', document_matrix.shape)


{'cache_id': '95638881ae385e4871a34626ee33af57b4b556a95ed6c019b7958030f08263ea',
 'origin': 'Fresh baseline document embeddings',
 'creator_run_id': 'baseline-fresh-5413aa92e9c1375fa06f',
 'matrix_sha256': '67f2a56d80a87918def7595ca487af4e64d4f4df92492c1a953742e53c7eea85',
 'matrix_shape': [10210, 2560]}

Saved document embedding matrix: (10210, 2560)


In [5]:
# Pipeline implementation; not called by this walkthrough.

document_attempts = pd.read_csv(RUN / 'document_embedding_attempts.csv')
display(document_attempts.groupby('status').size().rename('attempts').reset_index())
document_example = PROJECT / standard_config['document_embedding_example']
if document_example.is_file():
    show_saved_request(document_example, 'Saved document embedding request and response')
else:
    print('The document request example is unavailable; run provenance remains inspectable above.')


,status,attempts


The document request example is unavailable; run provenance remains inspectable above.


### Cosine retrieval and cache verification

Haystack’s in-memory document store uses the experiment’s document vectors. Inspect the implementation and the saved configuration/checksums. The index-construction function is not called by this walkthrough.


In [6]:
from shared_scoring.fresh_baseline import build_baseline_run
# This is the complete first-execution implementation, including the model requests
# and Haystack retrieval. It is called by ensure_baseline_study when results are absent.
display(Code(inspect.getsource(build_baseline_run), language='python'))
manifest = read_json(RUN / 'run_manifest.json')
contract, contract_hash = manifest['contract'], manifest['contract_sha256']
cache_manifest = {'matrix_sha256': contract['document_matrix_sha256']}
display({'experiment_document_cache_id': document_cache_id, 'contract_sha256': contract_hash,
         'scoring_sha256': contract['scoring_source_sha256']})


def build_baseline_run(original_config, project, directory, client=None, rows=None):
    """Create or resume the original two-path Haystack baseline in a separate directory."""
    os.environ.setdefault('HAYSTACK_TELEMETRY_ENABLED','False')
    try:
        from haystack import Document
        from haystack.document_stores.in_memory import InMemoryDocumentStore
        from haystack.components.retrievers.in_memory import InMemoryEmbeddingRetriever
    except ImportError as error:
        raise ImportError('A fresh baseline needs: python -m pip install -r requirements-runtime.txt') from error
    directory, project = Path(directory), Path(project)
    source = original_config['source_directory']
    settings = read_json(source/'config.json')
    cfg = settings['models_and_dataset']
    helpers = _module(source/'experiment_helpers.py','baseline_execution_helpers')
    scoring = _module(source/'scoring.py','baseline_original_metrics')
    docs, questions = prepare_baseline_inputs(source/'input',directory/'input',settings,original_config['baseline_scope'],rows)
    run_id = 'baseline-fresh-'+directory.name
    run = directory/'runs'/run_id
    run.mkdir(parents=True, exist_ok=True)
    client_settings = {**execution_settings(project)['retrieval'],
        'dimensions':cfg['embedding_dimensions'],'embedding_model':cfg['embedding_model'],
        'api_base':cfg['api_base'],'reranker_model':cfg['reranker_model'],'reranker_url':cfg['reranker_url'],
        'max_attempts':cfg['document_embedding_max_attempts'],'backoff_seconds':cfg['document_embedding_backoff_seconds']}
    client = client or RecordedOpenRouterClient(client_settings,directory/'requests')
    contract = {'settings':settings,'questions':questions,'documents':docs}
    contract_hash = fingerprint(contract)
    identity_path = directory/'batch_identity_v1.json'
    if identity_path.exists() and read_json(identity_path).get('contract_sha256')!=contract_hash:
        raise ValueError('Fresh baseline inputs changed; choose another runtime directory.')
    write_json(identity_path, {'run_id':run_id,'contract_sha256':contract_hash})
    example = directory/'document_embedding_example.json'
    scope_settings = execution_settings(project, include_scope=True)
    if original_config.get('eligible_baseline') and scope_settings.get('scope'):
        from .cached_scope import source_prefix
        source_cache, count, saved_matrix = source_prefix(scope_settings)
        source_corpus = pd.read_parquet(source_cache/'corpus.parquet').iloc[:count]
        positions = {key:i for i,key in enumerate(source_corpus.passage_id)}
        if any(doc['id'] not in positions for doc in docs):
            raise ValueError('An eligible baseline passage has no cached vector.')
        if any(doc['text'] != source_corpus.iloc[positions[doc['id']]].text for doc in docs):
            raise ValueError('Baseline text differs from the cached embedded passage.')
        matrix = np.asarray(saved_matrix[[positions[doc['id']] for doc in docs]]).copy()
        print(f'Reused {len(docs):,} cached passage embeddings for the eligible baseline; no passage embedding calls.',flush=True)
    else:
        vectors = []
        for start in range(0,len(docs),cfg['embedding_batch_size']):
            values,trace = client.embed([doc['text'] for doc in docs[start:start+cfg['embedding_batch_size']]], 'passage',cfg['embedding_dimensions'])
            vectors.append(values)
            if start==0:
                write_json(example,trace)
        matrix = np.concatenate(vectors)
    matrix_path = directory/'document_embeddings.npy'
    with matrix_path.open('wb') as handle:
        np.save(handle,matrix)
    doc_contract = fingerprint({'documents':docs,'model':cfg['embedding_model'],'dimensions':cfg['embedding_dimensions']})
    write_json(run/'document_embedding_origin.json', {'cache_id':doc_contract,'origin':'Fresh baseline document embeddings',
        'creator_run_id':run_id,'matrix_sha256':sha256_file(matrix_path),'

{'experiment_document_cache_id': '95638881ae385e4871a34626ee33af57b4b556a95ed6c019b7958030f08263ea',
 'contract_sha256': 'ea9a38a1e9253ce22214ae3e5393645fb04f5ea95eed881663f8b98ce84a1dd2',
 'scoring_sha256': 'd9089058d704f5e9362dddfc5f1ad6f8714afd4edceef2710e3b9d1d733c1157'}

## 5. Scoring implementation and definitions

| Metric | definition |
|---|---|
| **Recall@30 / Recall@5** | Unique known support IDs retrieved / all known support IDs |
| Hit@30 / Hit@5 | Any known support ID retrieved |
| MRR@5 | Reciprocal rank of first known support, else 0 |
| nDCG@5 | Binary DCG divided by ideal DCG for min(5, number of gold IDs) |
| Normalized EM | Any reference matches after normalization |
| **Token F1** | Maximum token multiset overlap F1 over references |
| Citation diagnostics | Known gold cited; count of cited IDs absent from selected five |
| Human rubric pass rate | Mean `correct` on questions with both successful arms and both reviews |

Normalization lowercases, removes single/grouped `[p_ID]` citations, removes punctuation and articles (`a`, `an`, `the`), then collapses whitespace. EM and F1 measure lexical agreement, not semantic correctness. Gold labels may be incomplete. Unreviewed human scores remain blank.

In [7]:
import re, math, string
from collections import Counter
import pandas as pd

def retrieval_scores(ids, gold):
    gold = set(gold)
    assert gold
    hits = [int(x in gold) for x in ids]
    return {'recall': len(set(ids) & gold) / len(gold), 'hit': float(any(hits)),
            'mrr': next((1/(i+1) for i,h in enumerate(hits) if h), 0.0),
            'ndcg': sum(h/math.log2(i+2) for i,h in enumerate(hits)) /
                    sum(1/math.log2(i+2) for i in range(min(len(ids),len(gold))))}

def normalize(text):
    text = text.lower()
    text = re.sub(
        r'\[\s*p_[a-f0-9]+(?:\s*,\s*p_[a-f0-9]+)*\s*\]',
        ' ',
        text,
    )
    text = text.translate(str.maketrans('', '', string.punctuation))
    return ' '.join(re.sub(r'\b(a|an|the)\b', ' ', text).split())

def answer_scores(answer, references):
    a = normalize(answer).split()
    scores = []
    for reference in references:
        b = normalize(reference).split()
        overlap = sum((Counter(a) & Counter(b)).values())
        scores.append(2*overlap/(len(a)+len(b)) if a or b else 1.0)
    return {'normalized_em': float(any(normalize(answer)==normalize(r) for r in references)),
            'token_f1': max(scores)}

def score_results(questions, results):
    lookup = {q['query_id']: q for q in questions}
    rows = []
    for record in results:
        q = lookup[record['query_id']]
        pool = retrieval_scores([d['id'] for d in q['candidates']], q['gold_ids'])
        for arm, result in record['arms'].items():
            row = {'query_id': q['query_id'], 'arm': arm, 'status': result['status'],
                   'recall_at_30': pool['recall'], 'hit_at_30': pool['hit']}
            if result['status'] == 'ok':
                ids = [d['id'] for d in result['top_five']]
                row.update({k+'_at_5': v for k,v in retrieval_scores(ids, q['gold_ids']).items()})
                row.update(answer_scores(result['answer'], q['answers']))
                row.update({k: result[k] for k in ['rank_seconds','generation_seconds','branch_seconds','end_to_end_seconds']})
                citations = set(re.findall(r'\[(p_[a-f0-9]+)\]', result['answer']))
                row['cites_known_gold'] = bool(citations & set(q['gold_ids']))
                row['invalid_citation_count'] = len(citations - set(ids))
            rows.append(row)
    return pd.DataFrame(rows)

def aggregate(scores, reviews=None):
    # Paired complete cases prevent different denominators between arms.
    counts = scores[scores.status.eq('ok')].groupby('query_id').arm.nunique()
    eligible = set(counts[counts.eq(2)].index)
    paired = scores[scores.query_id.isin(eligible)]
    records = []
    for arm in ['baseline','reranked']:
        frame = paired[paired.arm.eq(arm)]
        row = {'arm': arm, 'paired_questions': len(frame),
               'failed_questions': int(((scores.arm == arm)&(scores.status != 'ok')).sum())}
        for metric in ['recall_at_30','hit_at_30','recall_at_5','hit_at_5','mrr_at_5','ndcg_at_5','normalized_em','token_f1']:
            row[metric] = frame[metric].mean() if metric in frame else float('nan')
        for metric in ['rank_seconds','generation_seconds','end_to_end_seconds']:
            row['median_'+metric] = frame[metric].median() if metric in frame else float('nan')
        row.update(reviewed_pairs=0, rubric_pass_rate=float('nan'))
        if reviews is not None:
            reviewed = reviews.copy()
            reviewed['correct'] = pd.to_numeric(reviewed['correct'], errors='coerce')
            reviewed = reviewed[reviewed.query_id.astype(str).isin(eligible) & reviewed.correct.isin([0,1])]
            pairs = reviewed.groupby('query_id').arm.nunique()
            common = set(pairs[pairs.eq(2)].index)
            reviewed = reviewed[reviewed.query_id.isin(common)&reviewed.arm.eq(arm)]
            row.update(reviewed_pairs=len(reviewed), rubric_pass_rate=reviewed.correct.mean())
        records.append(row)
    return pd.DataFrame(records).set_index('arm')

def haystack_crosscheck(questions, results):
    from haystack import Document
    from haystack.components.evaluators import DocumentRecallEvaluator, DocumentMRREvaluator, AnswerExactMatchEvaluator
    lookup = {q['query_id']: q for q in questions}
    output = []
    for record in results:
        q = lookup[record['query_id']]
        for arm, result in record['arms'].items():
            if result['status'] != 'ok': continue
            args = {'ground_truth_documents': [[Document(id=x, content=x) for x in q['gold_ids']]],
                    'retrieved_documents': [[Document(id=d['id'], content=d['id']) for d in result['top_five']]]}
            output.append({'query_id': q['query_id'], 'arm': arm,
                'haystack_recall_at_5': DocumentRecallEvaluator(mode='multi_hit').run(**args)['score'],
                'haystack_mrr_at_5': DocumentMRREvaluator().run(**args)['score'],
                'haystack_raw_em': max(AnswerExactMatchEvaluator().run(ground_truth_answers=[a],
                                        predicted_answers=[result['answer']])['score'] for a in q['answers'])})
    return pd.DataFrame(output)

## 6. Retrieval, evidence selection and generation

Follow the question through query embedding, Haystack retrieval of thirty candidates, selection of five passages with or without reranking, and two separate generator requests.

The functions below show the actual Qwen/OpenRouter calls used by the executor. First execution runs the full batch through `ensure_baseline_study`; successful saved results and request responses are reused on subsequent executions. Replay only reads the completed run.


In [8]:
from shared_scoring.provider import rerank_payload

def embed_query(client, question):
    return client.embed([question], 'query', cfg['embedding_dimensions'])

def rerank_candidates(client, question, candidates):
    return client.rerank(question, candidates)

def generate_answer(client, query_id, arm, messages):
    payload = {'model': cfg['generator_model'], 'messages': messages,
               'temperature': cfg['temperature'], 'max_tokens': cfg['max_output_tokens'],
               **GENERATION_EXTRAS}
    previous = client.settings['max_attempts']
    client.settings['max_attempts'] = 1  # Original baseline generation cap.
    try:
        return client.request('generation', cfg['generator_base'] + '/chat/completions',
                              payload, cache_key=query_id + '-' + arm)
    finally:
        client.settings['max_attempts'] = previous

# The batch was executed/resumed in setup so every later section sees the same run.
# These exact request entry points are wired into build_baseline_run (shown above).
results = read_json(RUN / 'results.json')
questions_with_candidates = read_json(RUN / 'candidates.json')
print('Execution mode:', MODE, '| saved complete answers:', len(results) * len(ARMS))


Execution mode: execute | saved complete answers: 64


## 7. Completion and failures

A complete pair means two successful answers to the same question, one per RAG path. The original saved ledger shows 31 baseline questions plus the original effervescent diagnostic. The added IMEI diagnostic is displayed separately later and is not included in that ledger or the 31-question averages.

Attempted questions, successful answers, complete pairs and failed requests have different denominators. Failed answers remain unavailable; they do not become zero scores.


In [9]:
status_rows = []
for r in results:
    status_rows.append({'query_id': r['query_id'], 'scope': r['scope'],
        'retrieval_status': r.get('retrieval_status'),
        'baseline_status': r['arms']['baseline']['status'],
        'reranked_status': r['arms']['reranked']['status'],
        'successful_pair': all(r['arms'][a]['status'] == 'ok' for a in ARMS),
        'baseline_generation_attempted': r['arms']['baseline'].get('generation_attempted', False),
        'reranked_generation_attempted': r['arms']['reranked'].get('generation_attempted', False)})
status_table = pd.DataFrame(status_rows)
coverage_rows = []
for scope in ['baseline_31', 'diagnostic']:
    frame = status_table[status_table.scope.eq(scope)]
    coverage_rows.append({'scope': scope, 'planned_questions': 31 if scope == 'baseline_31' else 1,
        'attempted_questions': len(frame), 'successful_pairs': int(frame.successful_pair.sum()),
        'questions_without_successful_pair': int((~frame.successful_pair).sum()),
        'successful_baseline_answers': int(frame.baseline_status.eq('ok').sum()),
        'successful_reranked_answers': int(frame.reranked_status.eq('ok').sum()),
        'failed_or_blocked_arms': int((~frame.baseline_status.eq('ok')).sum() + (~frame.reranked_status.eq('ok')).sum())})
coverage = pd.DataFrame(coverage_rows)
failure_rows = [{'query_id': r['query_id'], 'scope': r['scope'], 'arm': a,
    'status': v['status'], 'failed_stage': v.get('failed_stage'),
    'generation_attempted': v.get('generation_attempted', False), 'error': v.get('error')}
    for r in results for a, v in r['arms'].items() if v['status'] != 'ok']
failures = pd.DataFrame(failure_rows, columns=['query_id', 'scope', 'arm', 'status',
                                            'failed_stage', 'generation_attempted', 'error'])
traces = [json.loads(p.read_text()) for p in sorted((RUN / 'requests').glob('*.json'))]
attempts = pd.DataFrame([{'request_id': t['request_id'], 'query_id': t['query_id'],
    'stage': t['stage'], 'arm': t.get('arm'), 'status': t['status'],
    'http_status': t.get('http_status'), 'elapsed_seconds': t.get('elapsed_seconds'),
    'error': t.get('error')} for t in traces])
display(coverage)
display(status_table)
display(failures if len(failures) else Markdown('No failed or blocked arms in this recorded batch.'))
display(attempts.groupby(['stage', 'status'], dropna=False).size().rename('requests').reset_index())
print('Document embedding attempts (index preparation, separate from question batch):')
display(document_attempts.groupby('status').size().rename('attempts').reset_index())


,scope,planned_questions,attempted_questions,successful_pairs,questions_without_successful_pair,successful_baseline_answers,successful_reranked_answers,failed_or_blocked_arms
0,baseline_31,31,31,31,0,31,31,0
1,diagnostic,1,1,1,0,1,1,0


,query_id,scope,retrieval_status,baseline_status,reranked_status,successful_pair,baseline_generation_attempted,reranked_generation_attempted
0,1088186,baseline_31,ok,ok,ok,True,True,True
1,1038185,baseline_31,ok,ok,ok,True,True,True
2,1086238,baseline_31,ok,ok,ok,True,True,True
3,1100277,baseline_31,ok,ok,ok,True,True,True
4,1029544,baseline_31,ok,ok,ok,True,True,True
5,1082398,baseline_31,ok,ok,ok,True,True,True
6,1055197,baseline_31,ok,ok,ok,True,True,True
7,1089273,baseline_31,ok,ok,ok,True,True,True
8,1085733,baseline_31,ok,ok,ok,True,True,True
9,1085627,baseline_31,ok,ok,ok,True,True,True


No failed or blocked arms in this recorded batch.

,stage,status,requests
0,generation,ok,64
1,query_embedding,ok,32
2,rerank_top30,ok,32


Document embedding attempts (index preparation, separate from question batch):


,status,attempts


## 8. Per-question scores and answers

Inspect retrieval coverage and answer overlap for each question. The scoring functions are defined in section 5. Questions that fail before retrieval remain explicit failed records with unavailable metrics.

Baseline averages use complete question pairs and exclude the diagnostic.


In [10]:
scorable_ids = {q['query_id'] for q in questions_with_candidates}
scorable_results = [r for r in results if r['query_id'] in scorable_ids]
scores = score_results(questions_with_candidates, scorable_results)
blocked_rows = [{'query_id': r['query_id'], 'arm': a, 'status': v['status']}
                for r in results if r['query_id'] not in scorable_ids
                for a, v in r['arms'].items()]
if blocked_rows:
    scores = pd.concat([scores, pd.DataFrame(blocked_rows)], ignore_index=True)
scores = scores.merge(pd.DataFrame([{'query_id': q['query_id'], 'scope': q['scope'],
    'question': q['query']} for q in batch_questions]), on='query_id', validate='many_to_one')
answers = pd.DataFrame([{'query_id': r['query_id'], 'arm': a,
    'answer': v.get('answer'), 'finish_reason': v.get('finish_reason'),
    'error': v.get('error'), 'failed_stage': v.get('failed_stage'),
    'selected_ids': [d['id'] for d in v.get('top_five', [])]}
    for r in results for a, v in r['arms'].items()])
per_question = scores.merge(answers, on=['query_id', 'arm'], validate='one_to_one')
metric_columns = ['recall_at_30', 'hit_at_30', 'recall_at_5', 'hit_at_5',
                  'mrr_at_5', 'ndcg_at_5', 'normalized_em', 'token_f1']
for col in metric_columns:
    if col not in per_question:
        per_question[col] = np.nan
scores = per_question
display(scores[['query_id', 'scope', 'arm', 'status'] + metric_columns].round(4))
# Full untruncated answers are shown in an expandable recorded-output table.
from html import escape
html_rows = ''.join('<tr><td>' + escape(str(r['query_id'])) + '</td><td>' + escape(r['arm']) +
    '</td><td>' + escape(str(r.get('answer') or r.get('error') or 'Unavailable')) + '</td></tr>'
    for r in answers.to_dict('records'))
from IPython.display import HTML
display(HTML('<details><summary>All 64 arm records: full answers or failures</summary>' +
    '<table><tr><th>Query ID</th><th>Arm</th><th>Answer / failure</th></tr>' + html_rows + '</table></details>'))


,query_id,scope,arm,status,recall_at_30,hit_at_30,recall_at_5,hit_at_5,mrr_at_5,ndcg_at_5,normalized_em,token_f1
0,1088186,baseline_31,baseline,ok,1.0,1.0,1.0,1.0,1.0000,1.0000,0.0,0.6667
1,1088186,baseline_31,reranked,ok,1.0,1.0,1.0,1.0,1.0000,1.0000,0.0,0.5366
2,1038185,baseline_31,reranked,ok,1.0,1.0,0.0,0.0,0.0000,0.0000,0.0,0.2333
3,1038185,baseline_31,baseline,ok,1.0,1.0,0.0,0.0,0.0000,0.0000,0.0,0.2250
4,1086238,baseline_31,baseline,ok,1.0,1.0,1.0,1.0,1.0000,1.0000,0.0,0.3091
5,1086238,baseline_31,reranked,ok,1.0,1.0,1.0,1.0,0.5000,0.6309,0.0,0.2975
6,1100277,baseline_31,reranked,ok,1.0,1.0,1.0,1.0,1.0000,1.0000,0.0,0.4681
7,1100277,baseline_31,baseline,ok,1.0,1.0,1.0,1.0,1.0000,1.0000,0.0,0.4783
8,1029544,baseline_31,baseline,ok,1.0,1.0,1.0,1.0,1.0000,1.0000,0.0,0.0769
9,1029544,baseline_31,reranked,ok,1.0,1.0,1.0,1.0,1.0000,1.0000,0.0,0.0282


Query ID,Arm,Answer / failure
1088186,baseline,"The weather in Texarkana, Arkansas on August 30 is expected to be 76 degrees with a chance of patchy rain (83% chance) and winds from the northeast at 13 mph. On August 31, the forecast is 75 degrees with light drizzle (96% chance of rain) and north winds at 9 mph [p_10af63d165c7c485aedb]."
1088186,reranked,"The current weather in Texarkana, Arkansas, as of August 30, is 76 degrees with a chance of patchy rain and 13 mph northeast winds. On August 31, the forecast is 75 degrees with light drizzle, a 96% chance of rain, and 9 mph north winds [p_10af63d165c7c485aedb]. The climate in Texarkana features hot summers, with July being the warmest month, and cold winters, with December being the coolest. The highest recorded temperature was 114 °F in 2007, and the lowest was 17 °F in 1990 [p_6f23571c07e9942f3b49]."
1038185,reranked,"The ideal credit card balance for a good credit score is one that keeps the credit utilization ratio at 20% or less of the available credit limit [p_cfc2436ca7fad93fec27]. For example, with a $10,000 credit limit, the balance should ideally not exceed $2,000 to maintain a favorable utilization rate [p_85a475075ec630d9edef]."
1038185,baseline,"The ideal credit card balance for a good credit score is one that keeps credit utilization at 20% or less of the credit limit [p_cfc2436ca7fad93fec27]. For example, with a $10,000 credit limit, carrying a balance of $3,000 or less is recommended for optimal credit scoring [p_85a475075ec630d9edef]. However, if the card does not report balances to credit bureaus (such as some true charge cards), utilization may not affect the score [p_b5fdfb1fff8942366ea0][p_2be27f8940733a0a7973]."
1086238,baseline,"Antibiotic resistance in bacteria is caused by the overuse and misuse of antibiotics, which promotes the development and spread of resistant properties. This can occur when antibiotics are used unnecessarily, such as for viral infections, leading to selective pressure that favors resistant bacteria. Additionally, bacteria can transfer resistance traits to other bacteria, accelerating the spread of resistance [p_595215c087c42dc57711][p_32a99c36bfc8d9cebdca][p_7e567c198906201dcfe1]."
1086238,reranked,"Antibiotic resistance in bacteria is primarily caused by the overuse and misuse of antibiotics, including taking them for viral infections when they are ineffective. This improper use allows harmless bacteria to develop resistance, which can be transferred to other bacteria, and creates opportunities for harmful bacteria to thrive. The repeated exposure to antibiotics also leads to the selection of resistant strains, such as those seen in staph infections [p_595215c087c42dc57711][p_32a99c36bfc8d9cebdca][p_519a18958ac4ed1fddd9][p_7e567c198906201dcfe1]."
1100277,reranked,"Gestation is the period of fetal development from conception until birth, also known as pregnancy, which lasts about nine months in humans [p_ded14ec74054e77e2a93][p_7e288e8d2ab7fccf7cbd]. It refers to the time a person or animal spends developing inside the mother's uterus before birth [p_74008bec93847cda08fb]."
1100277,baseline,"Gestation is the period of fetal development from conception until birth, also known as pregnancy, lasting approximately nine months in humans [p_ded14ec74054e77e2a93][p_7e288e8d2ab7fccf7cbd]. It refers to the time a person or animal spends developing inside the mother's uterus before birth [p_74008bec93847cda08fb]."
1029544,baseline,"Tracers can refer to multiple things based on the provided passages: 1. **A 2015 American action film** titled *Tracers*, directed by Daniel Benmayor and starring Taylor Lautner, Marie Avgeropoulos, Adam Rayner, and Rafi Gavron. It was released on March 20, 2015, and received mixed reviews, with a 27% rating on Rotten Tomatoes and a 45/100 on Metacritic [p_8c691bd8eb091d4fefd6][p_93d0cb28dd1a21b1eb61]. 2. **A character in the video game Overwatch** named Tracer (Lena Oxton), a former Overwatch agent known for 

## 9. Baseline results: 31 questions

Each automatic comparison uses the same complete baseline pairs. Completion counts and failures are reported separately. Retrieval coverage and answer overlap measure different things.

The new baseline is sampled from eligible labeled questions with seed 42. Support comes from dataset annotations; human review remains pending. The review rubric checks correctness and completeness, support from the supplied evidence, unsupported claims and appropriate uncertainty when evidence is insufficient.

Timing fields are retained here for inspection. **Experiment 4: Latency Tradeoff** measures response time separately.


In [11]:
review_path = RUN / 'manual_review.csv'
if not review_path.exists():
    review_rows = []
    for r in results:
        q = next(q for q in batch_questions if q['query_id'] == r['query_id'])
        for arm in ARMS:
            v = r['arms'][arm]
            if v['status'] == 'ok':
                review_rows.append({'query_id': r['query_id'], 'scope': r['scope'], 'arm': arm,
                    'question': q['query'], 'reference_answers': json.dumps(q['answers']),
                    'answer': v['answer'], 'context': v['context'], 'correct': '',
                    'evidence_used_correctly': '', 'same_answer_as_baseline': '',
                    'supporting_source_ids': '', 'rationale': '', 'reviewer': ''})
reviews = pd.read_csv(review_path, dtype={'query_id': str})
baseline_scores = scores[scores.scope.eq('baseline_31')].copy()
summary = aggregate(baseline_scores, reviews)
scorecard = summary.T.rename(columns=DISPLAY_NAMES)
scorecard['Delta (reranked − baseline)'] = (scorecard[DISPLAY_NAMES['reranked']] -
                                          scorecard[DISPLAY_NAMES['baseline']])
display(scorecard.round(4).fillna('Pending / unavailable'))
print('Denominator:', int(status_table[status_table.scope.eq('baseline_31')].successful_pair.sum()),
      'complete pairs out of 31 attempted baseline questions.')
print('Human review CSV:', review_path)

# Paired per-question changes: diagnostic excluded, failed pairs excluded.
paired_ids = set(status_table[status_table.scope.eq('baseline_31') & status_table.successful_pair].query_id)
paired_scores = scores[scores.query_id.isin(paired_ids)]
changes = pd.DataFrame(index=sorted(paired_ids))
changes.index.name = 'query_id'
for metric in metric_columns:
    pivot = paired_scores.pivot(index='query_id', columns='arm', values=metric)
    if all(a in pivot.columns for a in ARMS):
        changes[metric + '_delta'] = pivot['reranked'] - pivot['baseline']
display(changes.round(4))


arm,Embedding RAG,Embedding + reranking RAG,Delta (reranked − baseline)
paired_questions,31.0,31.0,0.0
failed_questions,0.0,0.0,0.0
recall_at_30,1.0,1.0,0.0
hit_at_30,1.0,1.0,0.0
recall_at_5,0.9032,0.9032,0.0
hit_at_5,0.9032,0.9032,0.0
mrr_at_5,0.7231,0.7645,0.0414
ndcg_at_5,0.7645,0.8001,0.0356
normalized_em,0.0,0.0,0.0
token_f1,0.3785,0.359,-0.0195


Denominator: 31 complete pairs out of 31 attempted baseline questions.
Human review CSV: .cache/qwen-runs/baseline/5413aa92e9c1375fa06f/runs/baseline-fresh-5413aa92e9c1375fa06f/manual_review.csv


,recall_at_30_delta,hit_at_30_delta,recall_at_5_delta,hit_at_5_delta,mrr_at_5_delta,ndcg_at_5_delta,normalized_em_delta,token_f1_delta
query_id,,,,,,,,
1029167,0.0,0.0,0.0,0.0,0.0000,0.0000,0.0,0.0301
1029544,0.0,0.0,0.0,0.0,0.0000,0.0000,0.0,-0.0488
1032146,0.0,0.0,0.0,0.0,0.0000,0.1228,0.0,0.0049
1033042,0.0,0.0,0.0,0.0,0.1667,0.1309,0.0,0.0000
1038185,0.0,0.0,0.0,0.0,0.0000,0.0000,0.0,0.0083
1040180,0.0,0.0,0.0,0.0,0.0000,0.0000,0.0,0.0000
1050417,0.0,0.0,0.0,0.0,0.0000,0.0000,0.0,-0.0494
1054544,0.0,0.0,0.0,0.0,0.0000,0.0000,0.0,0.0038
1055197,0.0,0.0,0.0,0.0,0.0000,0.0000,0.0,-0.0156


### Standard answer evaluation

Evaluation attribution: Hugging Face Evaluate’s SQuAD metric and RAGAS Faithfulness and Answer relevancy.

SQuAD normalized exact match and **Token F1** use **0–100 percentages**; the original answer scorer above uses **0–1**. **Annotated-support Recall@5** remains a separate retrieval metric on **0–1**. **Faithfulness** uses **0–1**; **Answer relevancy** is usually 0–1, with a mathematical cosine range of −1–1. RAGAS judges evidence support and question relevance; it is not human correctness.

Saved scores below use the same complete baseline pairs and each arm’s exact separate five passages. Truncated answers remain included. Generation failures and earlier judge failures remain inspectable. Diagnostic scores remain separate. Loading these tables never requests scoring or model calls.


In [12]:
# First-execution judging uses the exact newly generated baseline answers.
RUN_RAGAS_SCORING = MODE == 'execute' and os.environ.get('NOTEBOOK_RUN_JUDGE') == '1'
if RUN_RAGAS_SCORING:
    from shared_scoring import load_saved_inputs
    from shared_scoring.judge import run_ragas_evaluation
    from shared_scoring.fresh_baseline import prepare_baseline_scores
    first_inputs, _, _, _ = load_saved_inputs(standard_config)
    session.enable_calls()
    judge_action = await run_ragas_evaluation(first_inputs, ROOT, standard_config['run_id'],
        standard_config['score_directory'], standard_config['judge'], allow_model_calls=True)
    prepare_baseline_scores(standard_config)

# Load saved evaluation scores.
saved_standard = load_saved_scores(standard_config)
standard_inputs = saved_standard['inputs']
standard_diagnostics = saved_standard['diagnostic_inputs']
standard_audit = saved_standard['audit']
STANDARD_OUT = standard_config['score_directory']
display({'saved_run': standard_config['run_id'],
         'baseline_questions': len(baseline_ids),
         'complete_baseline_pairs': standard_audit['complete_baseline_pairs'],
         'saved_judge_outcomes': len(saved_standard['judge_scores'])})


1088186 baseline answer_relevancy: ok 0.7879529392011603 (11.0s)


1088186 baseline faithfulness: ok 1.0 (19.2s)


1088186 reranked answer_relevancy: ok 0.7995303559974372 (13.2s)


1088186 reranked faithfulness: ok 1.0 (26.7s)


1038185 reranked faithfulness: ok 0.5 (11.2s)


...

1084137 baseline faithfulness: ok 1.0 (16.4s)


1084137 baseline answer_relevancy: ok 0.9419695261841895 (11.6s)


1084137 reranked answer_relevancy: ok 0.9022043952859288 (13.5s)


1084137 reranked faithfulness: ok 1.0 (30.4s)


{'saved_run': 'baseline-fresh-5413aa92e9c1375fa06f',
 'baseline_questions': 31,
 'complete_baseline_pairs': 31,
 'saved_judge_outcomes': 124}

In [13]:
# Real framework entry points, shown without initializing either framework.
from shared_scoring.answers import load_squad_metric
from shared_scoring.judge import create_ragas_metrics, run_ragas_evaluation
from shared_scoring.actions import run_squad_scoring
display(Code(inspect.getsource(load_squad_metric), language='python'))
display(Code(inspect.getsource(create_ragas_metrics), language='python'))

# Optional scoring functions; not called by Run all.
def run_hugging_face_scoring(output_directory):
    """Compute deterministic SQuAD scores into a new directory."""
    return run_squad_scoring(standard_config, output_directory)

async def run_ragas_judge_explicitly(output_directory, allow_model_calls=False):
    """Evaluate saved answers only with explicit model-call authorization."""
    return await run_ragas_evaluation(
        standard_inputs, ROOT, standard_config['run_id'], output_directory,
        standard_config['judge'], allow_model_calls=allow_model_calls)


def load_squad_metric(metric_source='squad', cache_directory=None):
    """Load Hugging Face SQuAD explicitly; this is never called by replay."""
    if cache_directory:
        directory = Path(cache_directory).resolve()
        os.environ.setdefault('HF_HOME', str(directory / 'huggingface'))
        os.environ.setdefault('HF_MODULES_CACHE', str(directory / 'modules'))
        os.environ.setdefault('HF_EVALUATE_CACHE', str(directory / 'evaluate'))
    import evaluate
    return evaluate.load(str(metric_source), cache_dir=str(cache_directory) if cache_directory else None)

def create_ragas_metrics(client, embeddings, settings):
    """Construct the real RAGAS metrics only for explicit judge evaluation."""
    from ragas.llms import llm_factory
    from ragas.metrics.collections import Faithfulness, AnswerRelevancy
    llm = llm_factory(settings['judge_model'], provider='openai', adapter='instructor', client=client,
        temperature=settings['judge_temperature'], top_p=settings['judge_top_p'], max_tokens=settings['judge_max_tokens'],
        seed=settings['judge_seed'], max_retries=settings.get('structured_output_max_attempts', 1),
        system_prompt=settings['judge_system_prompt'])
    available = {'faithfulness': Faithfulness(llm=llm),
                 'answer_relevancy': AnswerRelevancy(llm=llm, embeddings=embeddings, strictness=settings['relevancy_strictness'])}
    unknown = set(settings['metrics']) - available.keys()
    if unknown:
        raise ValueError(f'Unsupported configured RAGAS metrics: {sorted(unknown)}')
    return {name: available[name] for name in settings['metrics']}

In [14]:
standard_scores = saved_standard['scores']
standard_summary = saved_standard['summary']
display(standard_summary.round(4))
display(standard_scores[['query_id', 'path', 'squad_exact_match_pct',
    'squad_token_f1_pct', 'faithfulness_0_1', 'answer_relevancy_0_1', 'truncated']])
display(pd.read_csv(STANDARD_OUT / 'scorer_discrepancies.csv'))
display(pd.DataFrame(read_json(STANDARD_OUT / 'scorer_edge_cases.json')))
print('SQuAD empty/empty Token F1 is 0; the original custom scorer returns 1.')
print('All reference answers participate in SQuAD multiple-reference scoring.')


,Metric,Scale,Successful matching pairs,Embedding RAG,Embedding + reranking RAG,Paired delta (reranking − embedding),Improved,Tied,Lower,Delta bootstrap 95% low,Delta bootstrap 95% high
0,Annotated-support Recall@5,0–1; existing retrieval metric,31,0.9032,0.9032,0.0000,0,31,0,0.0000,0.0000
1,SQuAD normalized exact match,0–100 percent,31,0.0000,0.0000,0.0000,0,31,0,0.0000,0.0000
2,Token F1,0–100 percent,31,37.8523,35.9012,-1.9512,7,8,16,-4.1769,0.0557
3,Faithfulness,0–1,31,0.9485,0.9275,-0.0210,4,22,5,-0.0807,0.0379
4,Answer relevancy,usually 0–1; cosine can be −1–1,31,0.7997,0.7885,-0.0112,7,17,7,-0.0316,0.0032


,query_id,path,squad_exact_match_pct,squad_token_f1_pct,faithfulness_0_1,answer_relevancy_0_1,truncated
0,1088186,Embedding RAG,0.0,66.666667,1.000000,0.787953,False
1,1088186,Embedding + reranking RAG,0.0,53.658537,1.000000,0.799530,False
2,1038185,Embedding + reranking RAG,0.0,23.333333,0.500000,0.944883,False
3,1038185,Embedding RAG,0.0,22.500000,1.000000,0.883825,False
4,1086238,Embedding RAG,0.0,30.909091,1.000000,0.983508,False
5,1086238,Embedding + reranking RAG,0.0,29.752066,1.000000,0.912520,False
6,1100277,Embedding + reranking RAG,0.0,46.808511,1.000000,0.897838,False
7,1100277,Embedding RAG,0.0,47.826087,1.000000,0.897838,False
8,1029544,Embedding RAG,0.0,7.692308,1.000000,0.873739,False
9,1029544,Embedding + reranking RAG,0.0,2.816901,0.750000,0.627226,False


,query_id,question,arm,path,answer,finish_reason,truncated,annotated_support_recall_at_5,existing_token_f1_0_1,existing_normalized_em_0_1,squad_answer,reference_answers,squad_exact_match_pct,squad_token_f1_pct,token_f1_delta_0_1,normalized_em_delta_0_1


,case,SQuAD Token F1 (0–100),custom Token F1 (0–1)
0,empty answer and empty reference,0,1


SQuAD empty/empty Token F1 is 0; the original custom scorer returns 1.
All reference answers participate in SQuAD multiple-reference scoring.


### IMEI diagnostic scoring — separate from the 31-question baseline averages

This overview displays the same saved IMEI records inspected in detail in section 11; it is not another run. Compare both paths for **1053665, “what is imei and where do i find it”**, using the unchanged **10,210-passage corpus**. Neither annotated support is present. The diagnostic’s new answers and deterministic scores are stored separately; the original 31-question averages are unchanged.


In [1]:
from pathlib import Path
import sys
PROJECT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "configs/experiment1.json").is_file())
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
from shared_scoring.imei_diagnostic import prepare_small, show_small, show_large_retrieval, show_large_answers, paths
from IPython.display import display, Markdown
import pandas as pd
import os
imei_small = prepare_small(PROJECT, allow_model_calls=os.environ.get("IMEI_ALLOW_MODEL_CALLS") == "1")
show_small(PROJECT, imei_small)


### IMEI diagnostic — 10,210-passage baseline corpus

**1053665: what is imei and where do i find it**

**0 of 2 annotated supporting passages are in this corpus.** Both retrieval paths therefore have annotated-support Recall@5 = 0. This separate diagnostic is excluded from the **31-question baseline averages**.

#### Embedding RAG

#### Embedding + reranking RAG

#### Embedding RAG

**Answer:**

#### Embedding + reranking RAG

**Answer:**

**Saved diagnostic scores:** SQuAD percentages. No new judge scores were requested for this small-corpus diagnostic.

,path,squad_token_f1_pct,squad_exact_match_pct,annotated_support_recall_at_5
0,Embedding RAG,8.333333,0.0,0.0
1,Embedding + reranking RAG,27.027027,0.0,0.0


## 10. Haystack evaluator cross-check

Compare the saved Haystack recall and MRR with the calculated retrieval scores. The evaluator uses passage IDs and multi-hit recall. Raw exact match is reported separately from normalized exact match.


In [17]:
# Load the saved Haystack cross-check.
crosscheck = pd.read_csv(RUN / 'haystack_crosscheck.csv', dtype={'query_id': str})
checked = scores.merge(crosscheck, on=['query_id', 'arm'], validate='one_to_one')
assert np.allclose(checked.recall_at_5, checked.haystack_recall_at_5)
assert np.allclose(checked.mrr_at_5, checked.haystack_mrr_at_5)
display(checked[checked.scope.eq('baseline_31')].groupby('arm')[
    ['haystack_recall_at_5', 'haystack_mrr_at_5', 'haystack_raw_em']].mean().round(4))


,haystack_recall_at_5,haystack_mrr_at_5,haystack_raw_em
arm,,,
baseline,0.9032,0.7231,0.0
reranked,0.9032,0.7645,0.0


## 11. Inspect the separate IMEI diagnostic

This detailed view revisits the same saved IMEI diagnostic shown in the scoring overview above. Trace **1053665** through the unchanged **10,210-passage baseline corpus**. It contains **0 of 2 annotated supports**. Compare each path’s passages, prompts, answers and diagnostic scores. No human correctness labels are assigned. Later notebooks show the saved larger-corpus recovery.


In [1]:
from pathlib import Path
import sys
PROJECT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "configs/experiment1.json").is_file())
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
from shared_scoring.imei_diagnostic import prepare_small, show_small, show_large_retrieval, show_large_answers, paths
from IPython.display import display, Markdown
import pandas as pd
imei_small = prepare_small(PROJECT, allow_model_calls=False)
display({"query_id": "1053665", "question": imei_small["retrieval"]["question"],
         "corpus_passages": 10210, "annotated_supports_present": 0, "annotated_supports_total": 2,
         "excluded_from_baseline_averages": True})


{'query_id': '1053665',
 'question': 'what is imei and where do i find it',
 'corpus_passages': 10210,
 'annotated_supports_present': 0,
 'annotated_supports_total': 2,
 'excluded_from_baseline_averages': True}

### Reused query embedding → exact baseline top 30

The existing Qwen query embedding is reused for local Haystack cosine retrieval. No passage or query embedding request is made. Inspect the thirty candidates below.

In [1]:
from pathlib import Path
import sys
PROJECT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "configs/experiment1.json").is_file())
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
from shared_scoring.imei_diagnostic import prepare_small, show_small, show_large_retrieval, show_large_answers, paths
from IPython.display import display, Markdown
import pandas as pd
imei_small = prepare_small(PROJECT, allow_model_calls=False)
display({"query_embedding_origin": imei_small["retrieval"]["query_embedding_origin"],
         "embedding_dimensions": 2560, "new_embedding_calls": 0})
display(pd.DataFrame(imei_small["retrieval"]["top30"])[["embedding_rank", "passage_id", "embedding_score", "text"]])


{'query_embedding_origin': '.cache/qwen-runs/cached-subset/84556ae3e8e650a930be/requests/98ee3e8bf63673324b3b29350c34bd4f707fefc43e2c0ee1ddb4090e8bd2e6f8/attempt-01.json',
 'embedding_dimensions': 2560,
 'new_embedding_calls': 0}

,embedding_rank,passage_id,embedding_score,text
0,1,p_5a45997fb6bc4a13b57a,0.423135,Wireless phone number; Account holder billing ...
1,2,p_dd5c85e0ab29e48c2997,0.392935,"To find your school's Site Code, please log in..."
2,3,p_998fb287cf11bc761fb7,0.391720,Simply Log On with your mobile phone number an...
3,4,p_19cdb207b1973a1a8aa7,0.387059,Create or access your user profile by entering...
4,5,p_82a8668aab88149e69cf,0.386614,"To find your school's Site Code, please log in..."
5,6,p_88166e9d68d3c5c8e8e9,0.380885,Enter your cell phone number and we will text ...
6,7,p_c4c226b47c00de128a55,0.375814,"Find a USB port. On laptops, they are typicall..."
7,8,p_d91ea8a15403bece17ba,0.372516,If you are using BlackBerry Device software ve...
8,9,p_77fc23c95b90963fcafa,0.368070,Please enter the customer number and access co...
9,10,p_3c44d60ed1aefa14ad10,0.363406,Internet Essentials. 1 Real-time protection a...


### IMEI: embedding top five and reranked top five

These are the separate diagnostic’s saved contexts from the unchanged 10,210-passage corpus. Neither contains an annotated support passage.

In [1]:
from pathlib import Path
import sys
PROJECT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "configs/experiment1.json").is_file())
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
from shared_scoring.imei_diagnostic import prepare_small, show_small, show_large_retrieval, show_large_answers, paths
from IPython.display import display, Markdown
import pandas as pd
show_small(PROJECT, prepare_small(PROJECT, allow_model_calls=False), part="retrieval")


### IMEI diagnostic — 10,210-passage baseline corpus

**1053665: what is imei and where do i find it**

**0 of 2 annotated supporting passages are in this corpus.** Both retrieval paths therefore have annotated-support Recall@5 = 0. This separate diagnostic is excluded from the **31-question baseline averages**.

#### Embedding RAG

#### Embedding + reranking RAG

### IMEI: exact contexts, prompts and two new diagnostic answers

The generator receives only each path’s own five passages. These diagnostic-only requests are cached separately from the original baseline run.

In [1]:
from pathlib import Path
import sys
PROJECT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "configs/experiment1.json").is_file())
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
from shared_scoring.imei_diagnostic import prepare_small, show_small, show_large_retrieval, show_large_answers, paths
from IPython.display import display, Markdown
import pandas as pd
show_small(PROJECT, prepare_small(PROJECT, allow_model_calls=False), part="answers")


### IMEI diagnostic — 10,210-passage baseline corpus

**1053665: what is imei and where do i find it**

**0 of 2 annotated supporting passages are in this corpus.** Both retrieval paths therefore have annotated-support Recall@5 = 0. This separate diagnostic is excluded from the **31-question baseline averages**.

#### Embedding RAG

**Answer:**

#### Embedding + reranking RAG

**Answer:**

**Saved diagnostic scores:** SQuAD percentages. No new judge scores were requested for this small-corpus diagnostic.

,path,squad_token_f1_pct,squad_exact_match_pct,annotated_support_recall_at_5
0,Embedding RAG,8.333333,0.0,0.0
1,Embedding + reranking RAG,27.027027,0.0,0.0


### IMEI: evaluation of the recorded diagnostic answers

SQuAD measures reference overlap, not evidence support. Annotated-support Recall@5 is zero in both paths because both annotated supports are absent from the corpus. These scores do not enter the 31-question averages.

In [1]:
from pathlib import Path
import sys
PROJECT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "configs/experiment1.json").is_file())
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
from shared_scoring.imei_diagnostic import prepare_small, show_small, show_large_retrieval, show_large_answers, paths
from IPython.display import display, Markdown
import pandas as pd
imei_small = prepare_small(PROJECT, allow_model_calls=False)
display(pd.read_csv(PROJECT / imei_small["score_file"])[["path", "squad_token_f1_pct", "squad_exact_match_pct", "annotated_support_recall_at_5"]])
print("Separate IMEI diagnostic; original 31-question averages unchanged. This display makes no model calls.")


Separate IMEI diagnostic; original 31-question averages unchanged. This display makes no model calls.


,path,squad_token_f1_pct,squad_exact_match_pct,annotated_support_recall_at_5
0,Embedding RAG,8.333333,0.0,0.0
1,Embedding + reranking RAG,27.027027,0.0,0.0


## 12. Validate the saved run

Validate recorded IDs, contexts, request coverage, model settings, paired denominators, and scoring provenance for the original saved batch (31 baseline questions plus effervescent). The separately saved IMEI diagnostic does not change that batch identity. Display the original saved audit without replacing its timestamps, manifest, results, or frozen snapshot.


In [23]:
assert len(results) == 32 and len(status_table[status_table.scope.eq('baseline_31')]) == 31
assert set(summary.paired_questions) == {len(paired_ids)}
assert len(attempts.request_id.unique()) == len(attempts)
for r in results:
    if r.get('retrieval_status') == 'ok':
        assert len(r['top30']) == len({d['id'] for d in r['top30']}) == 30
        for arm in ARMS:
            value = r['arms'][arm]
            if 'messages' in value:
                assert len(value['top_five']) == 5 and value['messages'][0]['content'] == SYSTEM
                assert {d['id'] for d in value['top_five']} <= {d['id'] for d in r['top30']}
                context = '\n\n'.join(f"[{d['id']}]\n{d['text']}" for d in value['top_five'])
                assert value['messages'][1]['content'] == f"Question: {r['query']}\n\nPassages:\n{context}\n\nAnswer:"
    for request_id in r['request_ids']:
        assert (RUN / 'requests' / (request_id + '.json')).exists()
for trace in traces:
    assert 'headers' not in trace
    assert trace['status'] in ('ok', 'error', 'started')
    if trace['stage'] == 'generation':
        p = trace['request_body']
        assert p['model'] == cfg['generator_model'] and p['temperature'] == 0
        assert all(p[k] == v for k, v in GENERATION_EXTRAS.items())
    if trace['status'] == 'ok':
        assert 'raw_response_json' in trace and 'raw_response_text' in trace

display(read_json(RUN / 'audit.json'))
print('Saved run validated; original artifacts were not rewritten.')


{'batch_question_count': 32,
 'baseline_attempted_questions': 31,
 'baseline_successful_pairs': 31,
 'human_review': 'pending',
 'judge_evaluation': 'explicit action; pending',
 'original_run_preserved': True}

Saved run validated; original artifacts were not rewritten.
